# Run Diff

2つの `run_id` を比較します。ロジック改修前後やデータ更新前後の回帰確認に使います。

In [ ]:
# Parameters
RUN_ID_A = "latest"
RUN_ID_B = "latest"
RUNS_ROOT = "research/data/runs"
METRIC = "return_to_dd"
DIFF_KEY = "case_name"
TOP = 50


In [ ]:
from IPython.display import Markdown, display
from research.src.store.trial_store import TrialStore
from research.src.store.views import diff, format_table

store = TrialStore(RUNS_ROOT)
run_id_a = store.resolve_run_id(RUN_ID_A)
run_id_b = store.resolve_run_id(RUN_ID_B)
rows_a = store.load_trials(run_id_a)
rows_b = store.load_trials(run_id_b)
diff_rows = diff(rows_a, rows_b, metric=METRIC, key=DIFF_KEY)
display(Markdown(f"## Diff: `{run_id_a}` → `{run_id_b}`"))
display(Markdown("```\n" + format_table(diff_rows[:TOP], ["key", "a", "b", "delta"]) + "\n```"))


In [ ]:
regressions = [row for row in diff_rows if isinstance(row.get("delta"), (int, float)) and row["delta"] < 0]
improvements = [row for row in diff_rows if isinstance(row.get("delta"), (int, float)) and row["delta"] > 0]
display(Markdown("## Regression summary"))
display(Markdown(
    f"- compared cases: {len(diff_rows)}\n"
    f"- improvements: {len(improvements)}\n"
    f"- regressions: {len(regressions)}"
))
display(Markdown("### Worst regressions"))
worst = sorted(regressions, key=lambda row: row["delta"])[:10]
display(Markdown("```\n" + format_table(worst, ["key", "a", "b", "delta"]) + "\n```"))
